# **Semantic search**

In [1]:
import chromadb

In [2]:
from pathlib import Path
path = Path(".").resolve().parent / "Collections"
print(path)

C:\Users\aliai\Desktop\M2 I2A\RAG Agentique\Backend\Collections


In [3]:
client = chromadb.PersistentClient(path=path)
collection = client.get_collection(
    name="jurisprudence"
)

## **Getting the docs from the collection**

In [4]:
stored = collection.get(include=["documents", "metadatas"])
keyword_documents = stored["documents"]
if not keyword_documents:
    raise ValueError("The Chroma collection contains no documents")
if not (len(keyword_documents) == len(stored["ids"]) == len(stored["metadatas"])):
    raise ValueError("Chroma returned misaligned documents, IDs, and metadata")

keyword_index = {
    "documents": keyword_documents,
    "ids": stored["ids"],
    "metadatas": stored["metadatas"],
}

## **Helping functions**

In [7]:
def _rank_cases(scores, candidate_indices, index, top_k):
    if top_k <= 0:
        raise ValueError("top_k must be a positive integer")

    best_chunks = {}
    for chunk_index in candidate_indices:
        metadata = index["metadatas"][chunk_index]
        source_id = metadata.get("source_id")
        if not source_id:
            raise ValueError(f"Chunk {index['ids'][chunk_index]} has no source_id metadata")

        score = float(scores[chunk_index])
        current = best_chunks.get(source_id)
        if current is None or score > current["score"]:
            best_chunks[source_id] = {
                "document": index["documents"][chunk_index],
                "score": score,
                "id": index["ids"][chunk_index],
                "source_id": source_id,
                "metadatas": metadata,
            }

    ranked_cases = sorted(
        best_chunks.values(),
        key=lambda result: (result["score"], result["source_id"]),
    )
    return ranked_cases[:top_k]

In [24]:
def semanticSearch( index: dict, query : str, n_results : int = 50, top_k : int = 5):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
    )
    
    scores = results["distances"][0]
    ids = results["ids"][0]
    candidate_indices = []
    scores_dict = [0]*len(index["ids"])
    for id in ids:
        i = index["ids"].index(id)
        candidate_indices.append(i)
        scores_dict[i] = scores[ids.index(id)]
    ranked_cases = _rank_cases(scores_dict, candidate_indices, index, top_k)
    return ranked_cases

In [50]:
from math import log2
from statistics import mean

K_VALUES = (5, 10, 20, 30)


def evaluate_strategy(search_function, method_name, evaluation_queries, index):
    per_query = []
    max_k = max(K_VALUES)

    for query_data in evaluation_queries:
        relevant_ids = set(query_data["relevant_source_ids"])
        if not relevant_ids:
            raise ValueError(f"{query_data['query_id']} has no relevance labels")

        ranked_results = search_function(
            query=query_data["query"],
            top_k=max_k,
            index=index,
            n_results=2000
        )
        ranked_ids = [result["source_id"] for result in ranked_results]
        if len(ranked_ids) != len(set(ranked_ids)):
            raise ValueError(f"{method_name} returned duplicate cases")

        row = {
            "method": method_name,
            "query_id": query_data["query_id"],
            "query": query_data["query"],
            "retrieved_source_ids": ranked_ids,
        }
        relevant_ranks = [
            rank for rank, source_id in enumerate(ranked_ids, start=1)
            if source_id in relevant_ids
        ]
        row["mrr@10"] = 1 / relevant_ranks[0] if relevant_ranks else 0.0

        for k in K_VALUES:
            found = sum(source_id in relevant_ids for source_id in ranked_ids[:k])
            row[f"precision@{k}"] = found / k
            row[f"recall@{k}"] = found / len(relevant_ids)

            dcg = sum(
                1 / log2(rank + 1)
                for rank, source_id in enumerate(ranked_ids[:k], start=1)
                if source_id in relevant_ids
            )
            ideal_count = min(len(relevant_ids), k)
            ideal_dcg = sum(
                1 / log2(rank + 1)
                for rank in range(1, ideal_count + 1)
            )
            row[f"ndcg@{k}"] = dcg / ideal_dcg if ideal_dcg else 0.0

        per_query.append(row)

    summary = {"method": method_name, "queries": len(per_query)}
    for metric in (
        "precision@5", "recall@5", "ndcg@5",
        "precision@10", "recall@10", "ndcg@10", "mrr@10",
        "precision@20", "recall@20", "ndcg@20", 
        "precision@30", "recall@30", "ndcg@30", 
    ):
        summary[metric] = mean(row[metric] for row in per_query)
    return summary, per_query

In [47]:
import json
from pathlib import Path

candidate_paths = [
    Path.cwd() / "keyword_evaluation.json",
    Path.cwd() / "Backend" / "Retreival" / "keyword_evaluation.json",
]
file_path = next((candidate for candidate in candidate_paths if candidate.is_file()), None)
if file_path is None:
    raise FileNotFoundError(
        "keyword_evaluation.json was not found from the current working directory"
    )

with file_path.open(encoding="utf-8") as file:
    evaluation_data = json.load(file)

queries = evaluation_data["queries"]

In [51]:
summary, details = evaluate_strategy(
        semanticSearch,
        "semantic",
        queries,
        keyword_index,
    )

In [52]:
details

[{'method': 'semantic',
  'query_id': 'q01',
  'query': 'Je cherche des affaires liées à la résiliation d’un contrat de travail.',
  'retrieved_source_ids': ['658096e83ea7c8c1120dd8dd',
   '6585e138638cf45b25ce5841',
   '6585e138638cf45b25ce584a',
   '6581ee0c3ea7c8c112520f6b',
   '6585e139638cf45b25ce584d',
   '658096e83ea7c8c1120dd9c4',
   '6581ee043ea7c8c112520df3',
   '658096ef3ea7c8c1120de53b',
   '658096e73ea7c8c1120dd85b',
   '658096e53ea7c8c1120dd488',
   '65876a8011b7b1b261cb1a2b',
   '6079a8829ba5988459c4dafd',
   '6079411b9ba5988459c40666',
   '658096ea3ea7c8c1120ddcc2',
   '6079a8829ba5988459c4daf5',
   '6079d63f9ba5988459c5a04b',
   '6253c83cbd3db21cbdd84ad3',
   '6597061af74364d4a5c862e5',
   '65ea13174fe60d50212512ed',
   '658096eb3ea7c8c1120dded8',
   '6079411c9ba5988459c406e5',
   '658096f03ea7c8c1120de62d',
   '659851a3b972daea122dd08b',
   '6581ee0d3ea7c8c112520f8f',
   '6079411c9ba5988459c406fb',
   '6253c839bd3db21cbdd84a69',
   '65970611f74364d4a5c861d4',
   '6597